Historical model comparison on already encoded data. Original test results for its tuned XGBoost: MAE 13,222.40 MAD, RMSE 22,128.52 MAD, R² 0.8971. This is NOT the deployed pipeline. Global preprocessing and validation reuse limit these scores. Use training/train.py for an explicit split and fresh fitted pipeline.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# For preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor


In [ ]:
# For modeling
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

# For data transformation
from scipy.stats import boxcox


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(ROOT / "data/processed/cars_encoded.csv")

# 9. TRAIN-TEST SPLIT
print("\n9. TRAIN-TEST SPLIT")





# ---------------- Prepare X and y -------------------------------
y = df['Price']                          # target on log‑scale
X = df.drop(columns=['Price'])  # predictors only



#Vérifier et supprimer les NaN dans X et y
nan_mask = X.isna().any(axis=1) | y.isna()
X = X[~nan_mask]
y = y[~nan_mask]

X.shape


In [ ]:
y.shape


In [ ]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30,  random_state=42, stratify=None
)
X_val,   X_test, y_val,  y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)


print(f"Training data: {X_train.shape[0]} samples")
print(f"Validation data: {X_val.shape[0]} samples")
print(f"Test data: {X_test.shape[0]} samples")


In [ ]:

# 10. MODEL TRAINING AND EVALUATION
print("\n10. MODEL TRAINING AND EVALUATION")

# Define evaluation function
def evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions on all sets
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    # Convert to original scale
    y_train_orig = y_train
    y_val_orig = y_val
    y_test_orig = y_test
    
    y_pred_train_orig = y_pred_train
    y_pred_val_orig = y_pred_val
    y_pred_test_orig = y_pred_test
    
    # Calculate metrics in original scale
    def calculate_metrics(y_true, y_pred):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        return rmse, mae, r2, mape
    
    # Training set metrics
    rmse_train, mae_train, r2_train, mape_train = calculate_metrics(y_train_orig, y_pred_train_orig)
    
    # Validation set metrics
    rmse_val, mae_val, r2_val, mape_val = calculate_metrics(y_val_orig, y_pred_val_orig)
    
    # Test set metrics
    rmse_test, mae_test, r2_test, mape_test = calculate_metrics(y_test_orig, y_pred_test_orig)
    
    # Print metrics
    print("\nTraining Set Metrics:")
    print(f"RMSE: {rmse_train:.2f} | MAE: {mae_train:.2f} | R²: {r2_train:.4f} | MAPE: {mape_train:.2f}%")
    
    print("\nValidation Set Metrics:")
    print(f"RMSE: {rmse_val:.2f} | MAE: {mae_val:.2f} | R²: {r2_val:.4f} | MAPE: {mape_val:.2f}%")
    
    print("\nTest Set Metrics:")
    print(f"RMSE: {rmse_test:.2f} | MAE: {mae_test:.2f} | R²: {r2_test:.4f} | MAPE: {mape_test:.2f}%")
    
    # Return all metrics for comparison
    return {
        'model': type(model).__name__,
        'train_rmse': rmse_train,
        'train_mae': mae_train,
        'train_r2': r2_train,
        'train_mape': mape_train,
        'val_rmse': rmse_val,
        'val_mae': mae_val,
        'val_r2': r2_val,
        'val_mape': mape_val,
        'test_rmse': rmse_test,
        'test_mae': mae_test,
        'test_r2': r2_test,
        'test_mape': mape_test,
        'model_obj': model
    }

# Define and train multiple models
models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, random_state=42),
    'LightGBM': lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=42)
}

# Train and evaluate each model
results = []
for name, model in models.items():
    print(f"\nEvaluating {name}...")
    try:
        result = evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test)
        results.append(result)
    except Exception as e:
        print(f"Error with {name}: {str(e)}")
        continue

# Create DataFrame of results
if results:
    results_df = pd.DataFrame(results)
    print("\nModel Comparison (Test Set R²):")
    print(results_df.sort_values('test_r2', ascending=False))
else:
    print("Aucun modèle n'a pu être évalué en raison d'erreurs.")
    exit()



In [ ]:
# 11. HYPERPARAMETER TUNING FOR BEST MODEL

param_grid = {
    'n_estimators': [400],
    'learning_rate': [0.03, 0.05, 0.07],
    'max_depth': [6, 8],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0, 0.5],
    'reg_lambda': [1, 2]
    
}
model = xgb.XGBRegressor(random_state=42)
# Grid search with 5-fold cross-validation on X_train
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

# Réentraînement sur X_train + X_val
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

best_model = grid_search.best_estimator_
best_model.fit(X_train_full, y_train_full)

# Évaluation sur tous les jeux
result = evaluate_model(
    best_model,
    X_train_full, X_val, X_test,
    y_train_full, y_val, y_test
)



In [ ]:
output_dir = ROOT / "training/runs/notebook"
output_dir.mkdir(parents=True, exist_ok=True)
import joblib

# Save the best_model to a file
joblib.dump(best_model, output_dir / "model_experiments.joblib", compress=3)
print("✅ Best model saved to 'best_xgb_model.joblib'")


In [ ]:
# Ajout: Learning curve dans l'espace inverse
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

def custom_scorer(estimator, X, y):
    """Calculates MSE in original space after inverse transform"""
    y_pred = estimator.predict(X)
    # Transform inverse pour y et y_pred
    y_orig = y
    y_pred_orig = y_pred
    return -mean_squared_error(y_orig, y_pred_orig)

train_sizes, train_scores, test_scores = learning_curve(
    best_model, 
    X_train_full, 
    y_train_full,
    cv=5,
    scoring=custom_scorer,  # Utilisation de notre scorer personnalisé
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

# Calcul des moyennes (négation car nous utilisons negative MSE)
train_scores_mean = -np.mean(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)
# Tracé de la courbe
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training score")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g", label="Cross-validation score")
plt.title("Learning Curve (In Original Scale)")
plt.xlabel("Training examples")
plt.ylabel("MSE (original scale)")
plt.legend(loc="best")
plt.grid()
plt.show()
